In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import json

In [3]:
# ============================================================
# CONFIG
# ============================================================
DATA_DIR  = Path("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/radar_data")
CKPT_DIR  = Path("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/checkpoints")
LOG_DIR   = Path("/store_new/mch/msclim/antoumos/R/develop/NOWPRECIP/new_project/logs")
CKPT_DIR.mkdir(exist_ok=True, parents= True)
LOG_DIR.mkdir(exist_ok=True, parents = True)

BATCH_SIZE  = 8
NUM_EPOCHS  = 50
LR          = 1e-4
NUM_WORKERS = 4
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device:     {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs:     {NUM_EPOCHS}")
print(f"LR:         {LR}")


Device:     cpu
Batch size: 8
Epochs:     50
LR:         0.0001


In [4]:
# ============================================================
# LOAD DATA
# ============================================================
print("\nLoading data...")
data  = np.load(DATA_DIR / "phase2_samples.npz")
X_all = np.log1p(data["X"]).astype(np.float32)
Y_all = np.log1p(data["Y"]).astype(np.float32)
print(f"X_all: {X_all.shape}")
print(f"Y_all: {Y_all.shape}")

# Load enriched metadata with split column
meta = pd.read_csv(DATA_DIR / "phase2_samples_meta_enriched.csv")
print(f"Meta: {meta.shape}")
print(meta["split"].value_counts())

# ============================================================
# SPLIT
# ============================================================
train_mask = meta["split"] == "train"
val_mask   = meta["split"] == "val"

X_train, Y_train = X_all[train_mask], Y_all[train_mask]
X_val,   Y_val   = X_all[val_mask],   Y_all[val_mask]

print(f"\nX_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")


Loading data...
X_all: (8602, 3, 501, 371)
Y_all: (8602, 1, 501, 371)
Meta: (8602, 13)
split
train    5984
test     1360
val      1258
Name: count, dtype: int64

X_train: (5984, 3, 501, 371)
X_val:   (1258, 3, 501, 371)


In [5]:
# ============================================================
# DATASET + DATALOADER
# ============================================================
PAD = (6, 7, 5, 6)  # (left, right, top, bottom) → 371→384, 501→512

class RadarDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = F.pad(self.X[i], PAD)
        y = self.Y[i]
        return x, y

train_dataset = RadarDataset(X_train, Y_train)
val_dataset   = RadarDataset(X_val,   Y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train batches: {len(train_loader)}")  # 5984/8 = 748
print(f"Val batches:   {len(val_loader)}")    # 1258/8 = 15

Train batches: 748
Val batches:   158


In [12]:
x_batch, y_batch = next(iter(train_loader))
print("x:", x_batch.shape)
print("y:", y_batch.shape)

with torch.no_grad():
    y_pred = model(x_batch.to(DEVICE))
print("pred:", y_pred.shape)

/users/antoumos/.conda/envs/pysteps/lib/python3.11/site-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


x: torch.Size([8, 3, 512, 384])
y: torch.Size([8, 1, 501, 371])
pred: torch.Size([8, 1, 501, 371])


In [9]:
# ============================================================
# MODEL
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = DoubleConv(in_ch, out_ch)
        self.pool = nn.MaxPool2d(2)
    def forward(self, x):
        skip = self.conv(x)
        return self.pool(skip), skip

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = DoubleConv(out_ch * 2, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, features=[64, 128, 256, 512]):
        super().__init__()
        self.enc1       = EncoderBlock(in_channels, features[0])
        self.enc2       = EncoderBlock(features[0], features[1])
        self.enc3       = EncoderBlock(features[1], features[2])
        self.enc4       = EncoderBlock(features[2], features[3])
        self.bottleneck = DoubleConv(features[3], features[3] * 2)
        self.dec4       = DecoderBlock(features[3] * 2, features[3])
        self.dec3       = DecoderBlock(features[3], features[2])
        self.dec2       = DecoderBlock(features[2], features[1])
        self.dec1       = DecoderBlock(features[1], features[0])
        self.final      = nn.Conv2d(features[0], out_channels, 1)

    def forward(self, x):
        x, s1 = self.enc1(x)
        x, s2 = self.enc2(x)
        x, s3 = self.enc3(x)
        x, s4 = self.enc4(x)
        x = self.bottleneck(x)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        x = self.final(x)
        x = F.relu(x)                  # non-negative predictions
        return x[:, :, 5:506, 6:377]  # crop back to (501, 371)

model     = UNet().to(DEVICE)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 31,037,633


In [ ]:
# ============================================================
# TRAINING LOOP
# ============================================================

# Track losses for logging
train_losses = []
val_losses   = []

best_val_loss    = float("inf")  # best val loss seen so far
patience         = 10            # stop if no improvement for 10 epochs
epochs_no_improve = 0            # counter

print("\nStarting training...")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Val Loss':>12} {'Best':>6}")
print("-" * 40)

for epoch in range(1, NUM_EPOCHS + 1):

    # --------------------------------------------------------
    # TRAINING
    # --------------------------------------------------------
    model.train()
    train_loss = 0.0

    for x_b, y_b in train_loader:
        x_b = x_b.to(DEVICE)
        y_b = y_b.to(DEVICE) 

        optimizer.zero_grad()
        pred = model(x_b)
        loss = criterion(pred, y_b)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # --------------------------------------------------------
    # VALIDATION
    # --------------------------------------------------------
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for x_b, y_b in val_loader:
            x_b = x_b.to(DEVICE)
            y_b = y_b[:, :, 5:506, 6:377].to(DEVICE)

            pred     = model(x_b)
            val_loss += criterion(pred, y_b).item()

    val_loss /= len(val_loader)

    # --------------------------------------------------------
    # LOGGING
    # --------------------------------------------------------
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    print(f"{epoch:>6} {train_loss:>12.6f} {val_loss:>12.6f} {'*' if is_best else ''}")

    # --------------------------------------------------------
    # CHECKPOINTING
    # --------------------------------------------------------
    # Save every epoch
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "train_loss":      train_loss,
        "val_loss":        val_loss,
    }, CKPT_DIR / f"checkpoint_epoch_{epoch:03d}.pt")

    # Save best model separately
    if is_best:
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "val_loss":    val_loss,
        }, CKPT_DIR / "best_model.pt")
        
    # Save loss log every epoch
    pd.DataFrame({
        "epoch":      list(range(1, epoch + 1)),
        "train_loss": train_losses,
        "val_loss":   val_losses,
    }).to_csv(LOG_DIR / "losses.csv", index=False)

    # --------------------------------------------------------
    # EARLY STOPPING
    # --------------------------------------------------------
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping at epoch {epoch} — no improvement for {patience} epochs")
        break

print(f"\nTraining complete. Best val loss: {best_val_loss:.6f}")